# Frame Extraction Pipeline → Manual Annotate on Roboflow
## Bradford Bulls — Logo Detection Training Data

### Pipeline
```
Video → Sample (1 FPS)
      → Filter quality (players visible, sharp)
      → Filter content (reject too-far wide shots, reject stage/not-on-pitch)
      → Score + de-dup + time diversity
      → Save full frames to Drive → Upload to Roboflow
      → YOU annotate manually on Roboflow → Train
```

### Key insights (why these filters matter)
- **Wide shots (quay quá xa) are not useful**: if you can't clearly see players, you definitely can't see kit sponsor logos. We reject these early using the *largest person bbox area ratio*.
- **Stage / interview / non-pitch frames are not useful**: even if YOLO detects people, these frames won't help logo-on-kit training. We reject these using a simple *pitch grass-green* heuristic.
- **Allow a configurable % of "non-relevant team"**: if you enable team relevance, you can enforce that at most `MAX_NON_RELEVANT_PCT` of the final selected frames are non-relevant.

> **Full frames, both teams, no cropping, no auto-annotation.**
> Runs on **Google Colab GPU**. All data on **Google Drive**.
> After Cell 1, **restart runtime** before continuing.

## 1. Install
> Run → **Restart runtime** → continue from Cell 2.

In [ ]:
!pip install -q ultralytics yt-dlp imagehash scikit-image pillow opencv-python-headless roboflow
print("\n✅ Done. RESTART RUNTIME now!")

## 2. Config
> `TEST_MODE = True` → 1000 frames only, select 20. Set `False` for full run.

### How to use the new settings
- **Reject wide shots**: increase `MIN_MAX_PERSON_AREA_RATIO` if you still see far-away frames; decrease it if you lose too many good gameplay frames.
- **Reject stage / not-on-pitch**: keep `ENABLE_PITCH_GREEN_FILTER = True`.
  - If it rejects too much (night games / different turf color), lower `MIN_PITCH_GREEN_RATIO`.
  - If stage/interview frames still slip through, raise `MIN_PITCH_GREEN_RATIO`.
- **% non-relevant team allowed**:
  - Keep `ENABLE_TEAM_RELEVANCE_FILTER = False` unless you have a reliable way to tag Bulls vs opponent.
  - If enabled, `MAX_NON_RELEVANT_PCT` caps how many non-relevant frames can appear in the final selection.

### Auditability
The output CSV includes `max_person_area_ratio`, `pitch_green_ratio`, and (if enabled) `bulls_color_ratio` + `is_relevant_team` so you can see why frames were kept/dropped.

In [ ]:
# ============================================================
# ⚠️ ALL CONFIG HERE
# ============================================================

VIDEO_URL = "https://www.youtube.com/watch?v=VjjO9Zmhdhk"

# Sampling
TARGET_FPS = 1              # Sample rate

# Basic player/sharpness filters
MIN_PLAYERS = 1             # At least 1 person visible
MIN_SHARPNESS = 0.30        # Frame sharpness threshold (0-1)

# "Too far" filter (reject wide shots where people are tiny)
# - Uses largest detected person bbox area / frame area
MIN_MAX_PERSON_AREA_RATIO = 0.03  # 3% of frame area

# "Stage / not on pitch" filter (heuristic)
# - Reject frames with too little grass-green pixels in bottom ROI
ENABLE_PITCH_GREEN_FILTER = True
PITCH_ROI_Y_START = 0.40          # analyze bottom 60% of image
MIN_PITCH_GREEN_RATIO = 0.08      # 8% of ROI pixels look like grass

# De-dup + diversity
DEDUP_HASH_THRESH = 6       # pHash distance — lower = stricter de-dup
TIME_BUCKET_SEC = 10        # Diversity: max frames per N-second window

# Optional: team relevance quota (keeps selection mostly "relevant")
# If you can't reliably detect Bulls vs opponent, keep this OFF.
ENABLE_TEAM_RELEVANCE_FILTER = False
MAX_NON_RELEVANT_PCT = 0.25        # allow at most 25% non-relevant frames
BULLS_COLOR_MIN_RATIO = 0.08       # within person crops, % pixels matching Bulls colors

# Bulls kit color heuristic (HSV ranges). Tune these if you enable team relevance.
# Each item is (lowerH, lowerS, lowerV, upperH, upperS, upperV)
BULLS_HSV_RANGES = [
    # Example reds (wrap-around reds may need 2 ranges)
    (0, 80, 60, 10, 255, 255),
    (170, 80, 60, 180, 255, 255),
    # Example dark/black
    (0, 0, 0, 180, 255, 50),
]

# Roboflow
ROBOFLOW_API_KEY = "API_KEY_HERE"  # ← Your key
PROJECT_NAME = "kit-sponsor-logos"

# ============================================================
# ⚠️ TEST vs PRODUCTION
# ============================================================
TEST_MODE = True

if TEST_MODE:
    MAX_FRAMES = 1000       # Process first 1000 frames
    TARGET_FRAMES = 20      # Select 20 frames
    print("🧪 TEST: 1000 frames → select 20")
else:
    MAX_FRAMES = None       # Full video
    TARGET_FRAMES = 400     # Select 400 frames
    print("🚀 PRODUCTION: full video → select 400 frames")

## 3. Setup

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU!")

from google.colab import drive
drive.mount("/content/drive")

import cv2, os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imagehash
from pathlib import Path
from collections import Counter
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO

DRIVE_ROOT = Path("/content/drive/MyDrive/Bradford_Bulls")
VIDEOS_DIR = DRIVE_ROOT / "videos"
FRAMES_DIR = DRIVE_ROOT / "selected_frames"
METADATA_DIR = DRIVE_ROOT / "metadata"

for d in [VIDEOS_DIR, FRAMES_DIR, METADATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Frames will be saved to: {FRAMES_DIR}")

## 4. Download / Load Video

In [ ]:
# 4. Load Video from Google Drive
# Tìm tất cả file .mp4 trong thư mục videos trên Drive, ưu tiên file mới nhất
existing_videos = sorted(VIDEOS_DIR.glob("*.mp4"), key=lambda f: f.stat().st_mtime, reverse=True)

if not existing_videos:
    print(f"❌ KHÔNG TÌM THẤY VIDEO!")
    print(f"👉 Vui lòng upload file video (.mp4) vào thư mục sau trên Google Drive:")
    print(f"   {VIDEOS_DIR}")
    print("\nSau khi upload xong, hãy chạy lại cell này.")
    raise FileNotFoundError("Thư mục videos đang trống.")

# Lấy video mới nhất được tìm thấy
VIDEO_PATH = existing_videos[0]
print(f"✅ Đã chọn video: {VIDEO_PATH.name}")

# Khởi tạo để lấy thông số video
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise ValueError(f"Không thể mở file video: {VIDEO_PATH}")

FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION = TOTAL_FRAMES / FPS
cap.release()

frame_interval = max(1, int(FPS / TARGET_FPS))
print(f"--- Thông tin video ---")
print(f"File: {VIDEO_PATH.name}")
print(f"Thời lượng: {DURATION/60:.1f} phút")
print(f"Độ phân giải: {W}x{H}")
print(f"FPS gốc: {FPS:.0f}")
print(f"Tổng số frame: {TOTAL_FRAMES:,}")
print(f"Trích xuất: 1 frame mỗi {frame_interval} frames (~{TARGET_FPS} FPS)")


## 5. Helpers

In [ ]:
def compute_frame_sharpness(frame):
    """Sharpness of the whole frame (multi-metric, 0-1)."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F).var()
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    ten = np.mean(gx**2 + gy**2)
    mean = gray.mean()
    nvar = gray.var() / (mean + 1e-6)
    return min(lap / 300, 1) * 0.40 + min(ten / 5000, 1) * 0.35 + min(nvar / 50, 1) * 0.25


def fmt_ts(sec):
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    return f"{h:02d}h{m:02d}m{s:02d}s" if h > 0 else f"{m:02d}m{s:02d}s"


def compute_pitch_green_ratio(frame_bgr, roi_y_start=0.40):
    """Return fraction of grass-green pixels in bottom ROI (0-1)."""
    h, w = frame_bgr.shape[:2]
    y0 = int(max(0, min(h - 1, roi_y_start * h)))
    roi = frame_bgr[y0:h, 0:w]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    # Broad grass-green range (tolerant to lighting).
    lower = np.array([35, 40, 40])
    upper = np.array([95, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)
    return float(mask.mean() / 255.0)


def compute_hsv_range_ratio(frame_bgr, hsv_ranges, boxes_xyxy):
    """Within person crops, compute ratio of pixels matching any HSV range."""
    if not hsv_ranges or boxes_xyxy is None or len(boxes_xyxy) == 0:
        return 0.0

    h, w = frame_bgr.shape[:2]
    ratios = []

    for (x1, y1, x2, y2) in boxes_xyxy:
        x1i, y1i = int(max(0, x1)), int(max(0, y1))
        x2i, y2i = int(min(w - 1, x2)), int(min(h - 1, y2))
        if x2i <= x1i or y2i <= y1i:
            continue
        crop = frame_bgr[y1i:y2i, x1i:x2i]
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

        mask_any = None
        for (lh, ls, lv, uh, us, uv) in hsv_ranges:
            lower = np.array([lh, ls, lv])
            upper = np.array([uh, us, uv])
            m = cv2.inRange(hsv, lower, upper)
            mask_any = m if mask_any is None else cv2.bitwise_or(mask_any, m)

        ratios.append(float(mask_any.mean() / 255.0) if mask_any is not None else 0.0)

    return float(max(ratios)) if ratios else 0.0


print("OK")

## 6. Sample + Detect Players + Score Quality
> Samples at 1 FPS, checks each frame for:
> - **Players visible** (YOLOv8 person detection, both teams)
> - **Sharpness** (not blurry)
> - Scores each frame for later selection

In [ ]:
model = YOLO("yolov8m.pt")
print(f"YOLOv8m loaded on {device}")

cap = cv2.VideoCapture(str(VIDEO_PATH))
frame_area = W * H

candidates = []  # frames that pass basic filters
batch_frames = []
batch_frame_nums = []

limit = MAX_FRAMES if TEST_MODE else TOTAL_FRAMES
pbar = tqdm(total=limit, desc="Scanning frames", unit="fr")

frame_num = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if TEST_MODE and MAX_FRAMES and frame_num >= MAX_FRAMES:
        break
    pbar.update(1)

    # Sample at TARGET_FPS
    if frame_num % frame_interval != 0:
        frame_num += 1
        continue

    batch_frames.append(frame)
    batch_frame_nums.append(frame_num)

    if len(batch_frames) < 16:
        frame_num += 1
        continue

    # Batch detect persons
    results = model.predict(batch_frames, classes=[0], conf=0.4, device=device, verbose=False)

    for fn, fr, res in zip(batch_frame_nums, batch_frames, results):
        n_persons = len(res.boxes) if res.boxes is not None else 0
        if n_persons < MIN_PLAYERS:
            continue

        sharpness = compute_frame_sharpness(fr)
        if sharpness < MIN_SHARPNESS:
            continue

        # Compute person sizes
        boxes_xyxy = []
        total_player_area = 0.0
        max_person_area_ratio = 0.0
        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            for i in range(len(xyxy)):
                x1, y1, x2, y2 = xyxy[i]
                boxes_xyxy.append((float(x1), float(y1), float(x2), float(y2)))
                area = max(0.0, (x2 - x1)) * max(0.0, (y2 - y1))
                total_player_area += area
                max_person_area_ratio = max(max_person_area_ratio, area / frame_area)

        # Too-far filter
        if max_person_area_ratio < MIN_MAX_PERSON_AREA_RATIO:
            continue

        # Stage / not-on-pitch filter (heuristic)
        pitch_green_ratio = compute_pitch_green_ratio(fr, roi_y_start=PITCH_ROI_Y_START)
        if ENABLE_PITCH_GREEN_FILTER and pitch_green_ratio < MIN_PITCH_GREEN_RATIO:
            continue

        player_coverage = min(total_player_area / frame_area, 1.0)

        # Optional: Bulls relevance score (heuristic by kit colors)
        bulls_color_ratio = compute_hsv_range_ratio(fr, BULLS_HSV_RANGES, boxes_xyxy)
        is_relevant_team = (bulls_color_ratio >= BULLS_COLOR_MIN_RATIO) if ENABLE_TEAM_RELEVANCE_FILTER else True

        # Score: more players + sharper + bigger players = better
        score = (
            sharpness * 0.40 +
            min(n_persons / 6, 1.0) * 0.30 +  # reward 2-6 players
            player_coverage * 0.30
        )

        timestamp = fn / FPS
        candidates.append({
            "frame_num": fn,
            "timestamp_sec": round(timestamp, 2),
            "timestamp_hms": fmt_ts(timestamp),
            "n_persons": n_persons,
            "sharpness": round(sharpness, 4),
            "player_coverage": round(player_coverage, 4),
            "max_person_area_ratio": round(float(max_person_area_ratio), 4),
            "pitch_green_ratio": round(float(pitch_green_ratio), 4),
            "bulls_color_ratio": round(float(bulls_color_ratio), 4),
            "is_relevant_team": bool(is_relevant_team),
            "score": round(score, 4),
        })

    batch_frames, batch_frame_nums = [], []
    frame_num += 1

# Remaining batch
if batch_frames:
    results = model.predict(batch_frames, classes=[0], conf=0.4, device=device, verbose=False)
    for fn, fr, res in zip(batch_frame_nums, batch_frames, results):
        n_persons = len(res.boxes) if res.boxes is not None else 0
        if n_persons < MIN_PLAYERS:
            continue

        sharpness = compute_frame_sharpness(fr)
        if sharpness < MIN_SHARPNESS:
            continue

        boxes_xyxy = []
        total_player_area = 0.0
        max_person_area_ratio = 0.0
        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            for i in range(len(xyxy)):
                x1, y1, x2, y2 = xyxy[i]
                boxes_xyxy.append((float(x1), float(y1), float(x2), float(y2)))
                area = max(0.0, (x2 - x1)) * max(0.0, (y2 - y1))
                total_player_area += area
                max_person_area_ratio = max(max_person_area_ratio, area / frame_area)

        if max_person_area_ratio < MIN_MAX_PERSON_AREA_RATIO:
            continue

        pitch_green_ratio = compute_pitch_green_ratio(fr, roi_y_start=PITCH_ROI_Y_START)
        if ENABLE_PITCH_GREEN_FILTER and pitch_green_ratio < MIN_PITCH_GREEN_RATIO:
            continue

        player_coverage = min(total_player_area / frame_area, 1.0)
        bulls_color_ratio = compute_hsv_range_ratio(fr, BULLS_HSV_RANGES, boxes_xyxy)
        is_relevant_team = (bulls_color_ratio >= BULLS_COLOR_MIN_RATIO) if ENABLE_TEAM_RELEVANCE_FILTER else True

        score = sharpness * 0.40 + min(n_persons / 6, 1.0) * 0.30 + player_coverage * 0.30
        timestamp = fn / FPS
        candidates.append({
            "frame_num": fn,
            "timestamp_sec": round(timestamp, 2),
            "timestamp_hms": fmt_ts(timestamp),
            "n_persons": n_persons,
            "sharpness": round(sharpness, 4),
            "player_coverage": round(player_coverage, 4),
            "max_person_area_ratio": round(float(max_person_area_ratio), 4),
            "pitch_green_ratio": round(float(pitch_green_ratio), 4),
            "bulls_color_ratio": round(float(bulls_color_ratio), 4),
            "is_relevant_team": bool(is_relevant_team),
            "score": round(score, 4),
        })

pbar.close()
cap.release()

candidates.sort(key=lambda x: x["score"], reverse=True)
print(f"\nCandidate frames: {len(candidates):,} (from {limit:,} scanned)")
if candidates:
    print(f"Score range: {candidates[-1]['score']:.3f} — {candidates[0]['score']:.3f}")
    print(f"max_person_area_ratio (min/max): {min(c['max_person_area_ratio'] for c in candidates):.4f} — {max(c['max_person_area_ratio'] for c in candidates):.4f}")
    print(f"pitch_green_ratio (min/max): {min(c['pitch_green_ratio'] for c in candidates):.4f} — {max(c['pitch_green_ratio'] for c in candidates):.4f}")
    if ENABLE_TEAM_RELEVANCE_FILTER:
        rel = sum(1 for c in candidates if c["is_relevant_team"])
        print(f"team relevance: {rel}/{len(candidates)} candidates marked relevant")

## 7. De-duplicate + Select Top N
> Remove near-duplicate frames (pHash), ensure time spread.

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))

selected = []
selected_hashes = []
time_bucket_counts = Counter()
MAX_PER_BUCKET = max(2, TARGET_FRAMES // (int(DURATION / TIME_BUCKET_SEC) + 1) + 2)

max_non_relevant = int(np.ceil(TARGET_FRAMES * MAX_NON_RELEVANT_PCT))
non_relevant_count = 0

print(f"Selecting top {TARGET_FRAMES} from {len(candidates)} candidates...")
print(f"  Max per {TIME_BUCKET_SEC}s window: {MAX_PER_BUCKET}")
if ENABLE_TEAM_RELEVANCE_FILTER:
    print(f"  Team relevance quota: allow ≤{max_non_relevant} non-relevant frames ({MAX_NON_RELEVANT_PCT:.0%})")

for meta in tqdm(candidates, desc="De-dup + select"):
    bucket = int(meta["timestamp_sec"] // TIME_BUCKET_SEC)
    if time_bucket_counts[bucket] >= MAX_PER_BUCKET:
        continue

    # Enforce quota: keep selection mostly relevant
    if ENABLE_TEAM_RELEVANCE_FILTER and (not meta.get("is_relevant_team", True)):
        if non_relevant_count >= max_non_relevant:
            continue

    cap.set(cv2.CAP_PROP_POS_FRAMES, meta["frame_num"])
    ret, frame = cap.read()
    if not ret:
        continue

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame_hash = imagehash.phash(Image.fromarray(frame_rgb))

    if any(frame_hash - h < DEDUP_HASH_THRESH for h in selected_hashes):
        continue

    meta["frame_hash"] = str(frame_hash)
    selected.append(meta)
    selected_hashes.append(frame_hash)
    time_bucket_counts[bucket] += 1

    if ENABLE_TEAM_RELEVANCE_FILTER and (not meta.get("is_relevant_team", True)):
        non_relevant_count += 1

    if len(selected) >= TARGET_FRAMES:
        break

cap.release()

print(f"\nSelected: {len(selected)} frames")
if selected:
    scores = [s["score"] for s in selected]
    print(f"Score: {min(scores):.3f} — {max(scores):.3f}")
    players = [s["n_persons"] for s in selected]
    print(f"Players/frame: {min(players)} — {max(players)} (avg {np.mean(players):.1f})")
    if ENABLE_TEAM_RELEVANCE_FILTER:
        nr = sum(1 for s in selected if not s.get("is_relevant_team", True))
        print(f"Non-relevant selected: {nr}/{len(selected)}")

## 8. Save Frames to Google Drive

In [ ]:
selected.sort(key=lambda x: x["timestamp_sec"])
cap = cv2.VideoCapture(str(VIDEO_PATH))
rows = []

for idx, meta in enumerate(tqdm(selected, desc="Saving frames")):
    cap.set(cv2.CAP_PROP_POS_FRAMES, meta["frame_num"])
    ret, frame = cap.read()
    if not ret:
        continue

    fname = f"frame_{idx+1:04d}_{meta['timestamp_hms']}_s{meta['score']:.2f}.jpg"
    cv2.imwrite(str(FRAMES_DIR / fname), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])

    rows.append({
        "id": idx + 1,
        "filename": fname,
        "frame_num": meta["frame_num"],
        "timestamp": meta["timestamp_hms"],
        "score": meta["score"],
        "sharpness": meta["sharpness"],
        "n_players": meta["n_persons"],
        "player_coverage": meta.get("player_coverage"),
        "max_person_area_ratio": meta.get("max_person_area_ratio"),
        "pitch_green_ratio": meta.get("pitch_green_ratio"),
        "bulls_color_ratio": meta.get("bulls_color_ratio"),
        "is_relevant_team": meta.get("is_relevant_team"),
        "video": VIDEO_PATH.name,
    })

cap.release()

df = pd.DataFrame(rows)
csv_path = METADATA_DIR / "selected_frames_index.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved {len(rows)} frames → {FRAMES_DIR}")
print(f"CSV → {csv_path}")

## 9. Preview

In [ ]:
frames = sorted(FRAMES_DIR.glob("frame_*.jpg"))
get_s = lambda p: float(p.stem.split("_s")[-1])
by_score = sorted(frames, key=get_s, reverse=True)

n = min(12, len(by_score))
rows = max(1, (n + 3) // 4)
fig, axes = plt.subplots(rows, 4, figsize=(20, 5*rows))
axes = np.array(axes).flatten()

for i, fp in enumerate(by_score[:n]):
    img = cv2.imread(str(fp))
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ts = fp.stem.split("_")[2]
    s = get_s(fp)
    axes[i].set_title(f"{ts} | score={s:.2f}", fontsize=9)
    axes[i].axis("off")
for j in range(n, len(axes)):
    axes[j].axis("off")

plt.suptitle(f"Top {n} frames — full frame, both teams, ready for annotation", fontsize=13)
plt.tight_layout()
plt.show()
print(f"Total: {len(frames)} frames saved")

## 10. Upload to Roboflow
> Frames uploaded **without any annotations**.
> You draw bounding boxes + assign classes manually on Roboflow.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace()

try:
    project = workspace.project(PROJECT_NAME)
    print(f"Connected: {project.name}")
except Exception:
    project = workspace.create_project(
        project_name="Kit Sponsor Logos",
        project_type="object-detection",
        project_license="MIT",
        annotation="kit-sponsor-logos",
    )
    print("Created project: Kit Sponsor Logos")

frames = sorted(FRAMES_DIR.glob("frame_*.jpg"))
success, errors = 0, 0

for fp in tqdm(frames, desc="Uploading"):
    try:
        project.upload(image_path=str(fp), split="train")
        success += 1
    except Exception as e:
        errors += 1
        if errors <= 3: print(f"  {e}")

print(f"\nDone! {success} uploaded, {errors} errors")
print(f"\n→ Annotate: https://app.roboflow.com/{workspace.url}/{PROJECT_NAME}/annotate")

## 11. Next Steps

1. **Annotate on Roboflow** — draw bounding boxes, assign class:

| ID | Class | ID | Class |
|----|-------|----|-------|
| 0 | aon_red | 11 | mna_cladding |
| 1 | aon_white | 12 | mna_support |
| 2 | atm_hospitality | 13 | paints_lacquers_yellow |
| 3 | cch_black | 14 | top_notch |
| 4 | cch_white | 15 | bartercard |
| 5 | chadlaw | 16 | floor_tonic |
| 6 | em_workwear | 17 | paints_lacquers_red |
| 7 | fairway_flooring | 18 | romantica_white |
| 8 | klg | 19 | romantica_black |
| 9 | mcp_away | 20 | acs_group |
| 10 | mcp_home | | |

2. **Generate dataset** — Roboflow: Versions → Generate (add augmentation)
3. **Train** — Roboflow Train (YOLOv8) or export YOLO format for local training